# Independent quantitative evaluation

This notebook keeps external checkpoint evaluation separate from the ablation runner. `evaluate_aging` downloads/validates AdaFace and DEX assets, runs one `.pt` checkpoint on a source/target pair, and reports ID-Source, ID-GT, and DEX age MAE. `evaluate_kid` is independent and compares two labeled `id_*/<age>.*` image trees. The low-level `load_quantitative_metrics` and `evaluate_aging_outputs` APIs remain available for an existing `prediction_manifest.csv`, where source and target must be the same person.

In [ ]:
from pathlib import Path

from src.quantitative_metrics import (
    DEFAULT_TARGET_AGE_STRENGTH_MAP,
    evaluate_aging,
    evaluate_aging_inference,
    evaluate_aging_inference_picking,
    evaluate_kid,
    prepare_quantitative_metrics,
)

## 1. Prepare metric assets

This downloads the official AdaFace repository/checkpoint and DEX files into one stable directory. They remain CPU-only during evaluation. Add `include_kid=True` only when running the separate KID section.

In [ ]:
EVALUATOR_ROOT = Path("/content/evaluation_models")
METRICS_CONFIG = prepare_quantitative_metrics(
    EVALUATOR_ROOT,
    device="cpu",
    include_kid=False,
)
print(METRICS_CONFIG)

## 2. Evaluate one `.pt` checkpoint (no KID)

The target image is the real longitudinal reference for ID-GT and DEX MAE. The generated image is saved under `OUTPUT_DIR`; the resulting manifest and CSV/JSON reports are reproducible.

In [ ]:
CHECKPOINT_PATH = Path("/content/checkpoints/case_00_full_epoch_010.pt")
SOURCE_IMAGE = Path("/content/data/sample/id_1144/35.png")
TARGET_IMAGE = Path("/content/data/sample/id_1144/53.jpg")
OUTPUT_DIR = Path("/content/aging_outputs/case_00_epoch_010")

result = evaluate_aging(
    checkpoint_path=CHECKPOINT_PATH,
    source_image=SOURCE_IMAGE,
    target_image=TARGET_IMAGE,
    source_age=35,
    target_age=53,
    metrics_config=METRICS_CONFIG,
    output_dir=OUTPUT_DIR,
    model_name="case_00_epoch_010",
    inference_config={"num_inference_steps": 50, "strength": 0.35, "seed": 2026},
    download_assets=False,
)
display(result["evaluation"]["summary"])
print(result["generated_path"])

## 3. Canonical inference with an exact target-age strength map (no picking)

This evaluates every requested target with the explicit canonical map. It does not search strengths; it preserves the exact policy used for the model comparison. Set `TARGET_IMAGES` to same-person ground-truth files when ID-GT is desired.

In [ ]:
TARGET_AGE_STRENGTH_MAP = {
    8: 0.33, 15: 0.31, 26: 0.04, 35: 0.22,
    44: 0.24, 55: 0.27, 65: 0.29,
}
TARGET_AGES = [8, 15, 26, 35, 44, 55, 65]
TARGET_IMAGES = {
    age: Path(f"/content/data/sample/id_1144/{age}.png")
    for age in TARGET_AGES
}
canonical = evaluate_aging_inference(
    checkpoint_path=CHECKPOINT_PATH,
    source_image=SOURCE_IMAGE,
    source_age=35,
    target_ages=TARGET_AGES,
    target_images=TARGET_IMAGES,
    target_age_strength_map=TARGET_AGE_STRENGTH_MAP,
    metrics_config=METRICS_CONFIG,
    output_dir=Path("/content/aging_outputs/canonical"),
    model_name="case_00_canonical",
    inference_config={"num_inference_steps": 40, "seed": 2026},
    download_assets=False,
)
display(canonical["evaluation"]["summary"])

## 4. Adaptive picking with positive/negative prompt assistance

This runs the adaptive diagnostic, compares the base and assisted candidate for each target, keeps the candidate with the smallest age error (identity cosine as tie-break), and evaluates only those selected images.

In [ ]:
prompt_assistance_config = {
    "enabled": True,
    "append_positive_support_prompt": True,
    "append_negative_prompt": True,
}
picked = evaluate_aging_inference_picking(
    checkpoint_path=CHECKPOINT_PATH,
    source_image=SOURCE_IMAGE,
    source_age=35,
    target_ages=TARGET_AGES,
    target_images=TARGET_IMAGES,
    target_age_strength_map=TARGET_AGE_STRENGTH_MAP,
    metrics_config=METRICS_CONFIG,
    output_dir=Path("/content/aging_outputs/picked"),
    model_name="case_00_picked",
    diagnostic_config={
        "num_inference_steps": 40,
        "text_reference_mode": "source_age",
        "age_guidance_scale": 3.0,
        "text_guidance_scale": 7.0,
        "image_guidance_scale": 1.5,
        "seed": 2026,
        "image_size": 256,
        "generate_assisted_prompt_variant": True,
        "prompt_assistance_config": prompt_assistance_config,
        "prompt_assistance_scale": 0.25,
        "negative_prompt_assistance_scale": 0.20,
    },
    download_assets=False,
)
display(picked["manifest"][["target_age", "selected_variant", "strength"]])
display(picked["evaluation"]["summary"])

## 5. KID only

`GENERATED_ROOT` and `REFERENCE_ROOT` must both contain matching labels such as `id_1144/53.png`. The function deterministically selects up to `num_ids` identities and `num_transitions` common target-age images.

In [ ]:
GENERATED_ROOT = Path("/content/generated_tree")
REFERENCE_ROOT = Path("/content/data/sample")

kid = evaluate_kid(
    generated_root=GENERATED_ROOT,
    reference_root=REFERENCE_ROOT,
    metrics_root=EVALUATOR_ROOT,
    num_ids=50,
    num_transitions=100,
    seed=2026,
)
print({key: kid[key] for key in ("kid_mean", "kid_std", "n_pairs")})
print(kid["summary_path"])